In [1]:
# =============================================================================
# Spatial Confusion Map — 5×5 Aggregated Model — Two Specific Days
# July 24, 2022  |  July 19, 2021
# model_ignition_agg5x5_smotebaseline.pkl
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import joblib
import glob
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

os.makedirs('figures', exist_ok=True)

# ── Configuration ─────────────────────────────────────────────────────────────
DATA_DIR = r'C:\Users\lambe\RU_Thesis_2026\ml_data'

DYNAMIC_FEATURES = [
    'temp_max', 'rh_min', 'vpd_mean', 'wind_speed_mean',
    'precip_sum', 'solar_rad_mj', 'rh_7d', 'temp_7d',
    'precip_30d', 'recovery_value'
]
STATIC_FEATURES = [
    'elevation_m', 'slope_deg', 'land_cover',
    'pop_density', 'road_proximity_m'
]
FEATURES        = DYNAMIC_FEATURES + STATIC_FEATURES
VAL_YEARS       = [2021, 2022]
DOY_START_HARD  = 91
DOY_END_HARD    = 310
COORD_PRECISION = 4

# Target days — same as pixel-level figure
TARGET_DAYS = ['20220724', '20210719']

COLORS = {
    'TN': '#a8d8ea',   # light blue
    'TP': '#e74c3c',   # bright red
    'FN': '#0d0d0d',   # black
    'FP': '#f39c12',   # orange
}
PLOT_ORDER = ['TN', 'FP', 'TP', 'FN']
ALPHAS     = {'TN': 0.75, 'FP': 0.90, 'TP': 0.95, 'FN': 1.00}
ZORDERS    = {'TN': 2,    'FP': 3,    'TP': 4,    'FN': 5}

# ── Load aggregated model ──────────────────────────────────────────────────────
print("Loading aggregated model...")
bundle_agg   = joblib.load(
    'models/model_ignition_agg5x5_smotebaseline.pkl')
model_agg    = bundle_agg['model']
CV_THRESHOLD = bundle_agg['cv_threshold']
BLOCK_SIZE   = bundle_agg['block_size']
AGG_METHOD   = bundle_agg['agg_method']

print(f"  CV threshold : {CV_THRESHOLD:.4f}")
print(f"  Block size   : {BLOCK_SIZE}×{BLOCK_SIZE} pixels")
print(f"  Agg method   : {AGG_METHOD}")

# ── Load validation data ───────────────────────────────────────────────────────
print("\nLoading validation data with coordinates...")

KEEP_COLS = ['date', 'longitude', 'latitude',
             'target_ignition'] + FEATURES

val_dfs = []
for year in VAL_YEARS:
    for f in sorted(glob.glob(
            os.path.join(DATA_DIR, f'ML_val_{year}??.csv'))):
        try:
            df = pd.read_csv(f, usecols=KEEP_COLS, on_bad_lines='skip')
            val_dfs.append(df)
        except Exception as e:
            print(f"  WARNING: {os.path.basename(f)}: {e}")

df_val = pd.concat(val_dfs, ignore_index=True)
df_val['date'] = df_val['date'].astype(int).astype(str)
df_val['doy']  = pd.to_datetime(
    df_val['date'], format='%Y%m%d', errors='coerce').dt.dayofyear
df_val = df_val[(df_val['doy'] >= DOY_START_HARD) &
                (df_val['doy'] <= DOY_END_HARD)].drop(columns='doy')
df_val = df_val.dropna(
    subset=FEATURES + ['target_ignition', 'longitude', 'latitude'])
df_val['target_ignition'] = df_val['target_ignition'].astype(int)
df_val['land_cover']      = df_val['land_cover'].astype(int)

print(f"  Validation pixel rows: {len(df_val):,}")

# ── Aggregate all validation data into 5×5 blocks ─────────────────────────────
print("\nAggregating pixels into 5×5 blocks...")

CONTINUOUS_FEATS = [f for f in FEATURES if f != 'land_cover']
CATEGORICAL_FEAT = 'land_cover'

pixel_spacing = df_val['longitude'].diff().abs()
pixel_spacing = pixel_spacing[pixel_spacing > 0].median()
block_spacing = pixel_spacing * BLOCK_SIZE

print(f"  Pixel spacing : {pixel_spacing:.4f}°")
print(f"  Block spacing : {block_spacing:.4f}°  "
      f"(~{block_spacing * 111:.0f} km)")

df_val['lon_r']     = df_val['longitude'].round(COORD_PRECISION)
df_val['lat_r']     = df_val['latitude'].round(COORD_PRECISION)
df_val['block_lon'] = np.floor(
    df_val['lon_r'] / block_spacing).astype(int)
df_val['block_lat'] = np.floor(
    df_val['lat_r'] / block_spacing).astype(int)
df_val['block_id']  = (df_val['date'] + '_' +
                        df_val['block_lon'].astype(str) + '_' +
                        df_val['block_lat'].astype(str))

agg_dict = {feat: AGG_METHOD for feat in CONTINUOUS_FEATS}
agg_dict[CATEGORICAL_FEAT]  = lambda x: x.mode().iloc[0]
agg_dict['target_ignition'] = 'max'
agg_dict['date']            = 'first'
agg_dict['block_lon']       = 'first'
agg_dict['block_lat']       = 'first'
agg_dict['lon_r']           = 'mean'
agg_dict['lat_r']           = 'mean'

df_agg = df_val.groupby('block_id').agg(agg_dict).reset_index()
df_agg['land_cover']      = df_agg['land_cover'].astype(int)
df_agg['target_ignition'] = df_agg['target_ignition'].astype(int)

print(f"  Total blocks  : {len(df_agg):,}")
print(f"  Fire blocks   : {(df_agg['target_ignition']==1).sum():,}")
print(f"  No-fire blocks: {(df_agg['target_ignition']==0).sum():,}")

# ── Run predictions on all aggregated blocks ───────────────────────────────────
print("\nRunning predictions on aggregated blocks...")

X_agg        = df_agg[FEATURES].values
y_true_agg   = df_agg['target_ignition'].values
y_prob_agg   = model_agg.predict_proba(X_agg)[:, 1]
y_pred_agg   = (y_prob_agg >= CV_THRESHOLD).astype(int)

df_agg['y_true'] = y_true_agg
df_agg['y_pred'] = y_pred_agg

def conf_class(row):
    if   row['y_true'] == 0 and row['y_pred'] == 0: return 'TN'
    elif row['y_true'] == 1 and row['y_pred'] == 1: return 'TP'
    elif row['y_true'] == 1 and row['y_pred'] == 0: return 'FN'
    else:                                             return 'FP'

df_agg['conf_class'] = df_agg.apply(conf_class, axis=1)

print(f"\n  Full validation confusion (block level):")
for cls in ['TN', 'TP', 'FN', 'FP']:
    n = (df_agg['conf_class'] == cls).sum()
    print(f"    {cls}: {n:,}")

# ── Verify target days exist in aggregated data ────────────────────────────────
print(f"\nVerifying target days in aggregated data...")
for day in TARGET_DAYS:
    n = (df_agg['date'] == day).sum()
    if n == 0:
        raise ValueError(
            f"Day {day} not found in aggregated data. "
            f"Available dates range: "
            f"{df_agg['date'].min()} – {df_agg['date'].max()}")
    n_tp = ((df_agg['date'] == day) &
            (df_agg['conf_class'] == 'TP')).sum()
    n_fp = ((df_agg['date'] == day) &
            (df_agg['conf_class'] == 'FP')).sum()
    n_fn = ((df_agg['date'] == day) &
            (df_agg['conf_class'] == 'FN')).sum()
    n_tn = ((df_agg['date'] == day) &
            (df_agg['conf_class'] == 'TN')).sum()
    n_fire = n_tp + n_fn
    print(f"  {day}: {n:,} blocks  |  "
          f"fires={n_fire}  TP={n_tp}  FP={n_fp}  "
          f"FN={n_fn}  TN={n_tn:,}")

# ── Prepare per-day data ───────────────────────────────────────────────────────
day_data = {}
for day in TARGET_DAYS:
    df_d = df_agg[df_agg['date'] == day].copy()

    n_tp   = (df_d['conf_class'] == 'TP').sum()
    n_fp   = (df_d['conf_class'] == 'FP').sum()
    n_fn   = (df_d['conf_class'] == 'FN').sum()
    n_tn   = (df_d['conf_class'] == 'TN').sum()
    n_fire = n_tp + n_fn

    recall = n_tp / max(n_tp + n_fn, 1)
    fpr    = n_fp / max(n_fp + n_tn, 1)
    prec   = n_tp / max(n_tp + n_fp, 1)

    day_data[day] = {
        'df'    : df_d,
        'n_tp'  : n_tp,
        'n_fp'  : n_fp,
        'n_fn'  : n_fn,
        'n_tn'  : n_tn,
        'n_fire': n_fire,
        'recall': recall,
        'fpr'   : fpr,
        'prec'  : prec,
    }

# ── Compute shared dot size from full domain extent ────────────────────────────
all_lons = pd.concat([d['df']['lon_r'] for d in day_data.values()])
all_lats = pd.concat([d['df']['lat_r'] for d in day_data.values()])
LON_RANGE = all_lons.max() - all_lons.min()
LAT_RANGE = all_lats.max() - all_lats.min()

FIG_W, FIG_H   = 20, 10
DPI            = 300
AXES_W_EACH    = (FIG_W / 2) * 0.78
AXES_H         = FIG_H * 0.78

PX_PER_DEG_X  = (AXES_W_EACH * DPI) / LON_RANGE
PX_PER_DEG_Y  = (AXES_H      * DPI) / LAT_RANGE
BLOCK_PX      = block_spacing * min(PX_PER_DEG_X, PX_PER_DEG_Y)
POINTS_PER_PX = DPI / 72
S_BASE        = (BLOCK_PX / POINTS_PER_PX) ** 2 * 0.72

S_SIZES = {
    'TN': S_BASE * 0.85,
    'FP': S_BASE * 1.1,
    'TP': S_BASE * 1.3,
    'FN': S_BASE * 1.6,
}

print(f"\n  Block dot size (s): {S_BASE:.1f}")

# ── Shared axis limits ─────────────────────────────────────────────────────────
LON_MIN = float(all_lons.min()) - block_spacing
LON_MAX = float(all_lons.max()) + block_spacing
LAT_MIN = float(all_lats.min()) - block_spacing * 0.6
LAT_MAX = float(all_lats.max()) + block_spacing * 0.6

# ── Two-panel figure ───────────────────────────────────────────────────────────
print(f"\nGenerating two-panel aggregated confusion map...")

fig, axes = plt.subplots(1, 2, figsize=(FIG_W, FIG_H))
fig.patch.set_facecolor('white')
fig.subplots_adjust(wspace=0.08)

for ax, day in zip(axes, TARGET_DAYS):
    info     = day_data[day]
    df_d     = info['df']
    date_obj = datetime.strptime(day, '%Y%m%d')
    date_str = date_obj.strftime('%B %d, %Y')
    date_doy = date_obj.timetuple().tm_yday

    for cls in PLOT_ORDER:
        subset = df_d[df_d['conf_class'] == cls]
        if len(subset) == 0:
            continue
        ax.scatter(
            subset['lon_r'], subset['lat_r'],
            c          = COLORS[cls],
            s          = S_SIZES[cls],
            alpha      = ALPHAS[cls],
            linewidths = 0,
            rasterized = True,
            zorder     = ZORDERS[cls],
            marker     = 's'
        )

    ax.set_xlim(LON_MIN, LON_MAX)
    ax.set_ylim(LAT_MIN, LAT_MAX)

    ax.set_xlabel('Longitude (°W)', fontsize=10)
    ax.set_ylabel('Latitude (°N)',  fontsize=10)
    ax.set_title(
        f'{date_str}  (DOY {date_doy})\n'
        f'TP = {info["n_tp"]}  '
        f'FN = {info["n_fn"]}  '
        f'FP = {info["n_fp"]:,}  '
        f'TN = {info["n_tn"]:,}',
        fontsize=11, fontweight='bold', pad=10
    )
    ax.tick_params(labelsize=9)
    ax.grid(color='#e8e8e8', linewidth=0.4, alpha=0.7, zorder=1)
    ax.set_axisbelow(True)
    ax.spines[['top', 'right']].set_visible(False)

    # Per-panel stats box
    stats_text = (
        f"Fire blocks  : {info['n_fire']}\n"
        f"Recall       : {info['recall']:.3f}\n"
        f"Precision    : {info['prec']:.4f}\n"
        f"FPR          : {info['fpr']:.3f}"
    )
    ax.text(
        0.985, 0.985, stats_text,
        transform           = ax.transAxes,
        fontsize            = 8.5,
        verticalalignment   = 'top',
        horizontalalignment = 'right',
        fontfamily          = 'monospace',
        bbox                = dict(boxstyle='round,pad=0.5',
                                   facecolor='white',
                                   edgecolor='#cccccc',
                                   alpha=0.95)
    )

# ── Shared legend ──────────────────────────────────────────────────────────────
legend_labels = {
    'TN': 'True Negative  — No fire block, correctly predicted',
    'TP': 'True Positive  — Fire block correctly detected',
    'FP': 'False Positive — No fire block, incorrectly flagged',
    'FN': 'False Negative — Fire block missed',
}
legend_patches = [
    mpatches.Patch(
        facecolor = COLORS[cls],
        edgecolor = '#888888' if cls == 'TN' else 'none',
        linewidth = 0.5,
        label     = legend_labels[cls])
    for cls in ['TN', 'TP', 'FP', 'FN']
]
fig.legend(
    handles        = legend_patches,
    fontsize       = 9,
    loc            = 'lower center',
    ncol           = 4,
    framealpha     = 0.95,
    edgecolor      = '#cccccc',
    bbox_to_anchor = (0.5, -0.04),
    title          = (f'Prediction outcome  |  '
                      f'Block = {BLOCK_SIZE}×{BLOCK_SIZE} pixels  '
                      f'(≈{block_spacing*111:.0f} km)'),
    title_fontsize = 9,
)

# ── Overall title ──────────────────────────────────────────────────────────────
fig.suptitle(
    f'Spatial Confusion Maps — 5×5 Aggregated Ignition Model\n'
    f'XGBoost  |  Threshold = {CV_THRESHOLD:.3f}  |  '
    f'Block resolution ≈ {block_spacing*111:.0f} km',
    fontsize=13, fontweight='bold', y=1.02
)
fig.text(
    0.5, -0.08,
    'Ontario wildfire ignition model  |  '
    '5×5 Spatial Aggregation  |  XGBoost  |  '
    'Source: Canadian National Fire Database (CNFDB)',
    ha='center', fontsize=8, color='#888888', style='italic'
)

plt.tight_layout()

fig.savefig('figures/fig_confusion_map_agg5x5_two_days.png',
            dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig('figures/fig_confusion_map_agg5x5_two_days.pdf',
            bbox_inches='tight', facecolor='white')

print("  Saved: figures/fig_confusion_map_agg5x5_two_days.png")
print("  Saved: figures/fig_confusion_map_agg5x5_two_days.pdf")
plt.close(fig)

# ── Summary ────────────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("SUMMARY — AGGREGATED TWO-DAY CONFUSION MAP")
print(f"{'='*60}")
print(f"  Model     : model_ignition_agg5x5_smotebaseline.pkl")
print(f"  Threshold : {CV_THRESHOLD:.4f}")
print(f"  Block size: {BLOCK_SIZE}×{BLOCK_SIZE} pixels  "
      f"(≈{block_spacing*111:.0f} km)")
print(f"\n  {'Day':<14} {'Fire blk':>9} {'TP':>5} {'FN':>5} "
      f"{'FP':>8} {'Recall':>8} {'FPR':>8}")
print(f"  {'─'*62}")
for day, info in day_data.items():
    date_str = datetime.strptime(day, '%Y%m%d').strftime('%b %d, %Y')
    print(f"  {date_str:<14} {info['n_fire']:>9} {info['n_tp']:>5} "
          f"{info['n_fn']:>5} {info['n_fp']:>8,} "
          f"{info['recall']:>8.3f} {info['fpr']:>8.3f}")

print(f"\nLaTeX:")
print(f"  \\includegraphics[width=\\textwidth]"
      f"{{figures/fig_confusion_map_agg5x5_two_days}}")

Loading aggregated model...
  CV threshold : 0.7398
  Block size   : 5×5 pixels
  Agg method   : median

Loading validation data with coordinates...
  Validation pixel rows: 5,221,040

Aggregating pixels into 5×5 blocks...
  Pixel spacing : 0.1000°
  Block spacing : 0.5000°  (~55 km)
  Total blocks  : 247,280
  Fire blocks   : 1,112
  No-fire blocks: 246,168

Running predictions on aggregated blocks...

  Full validation confusion (block level):
    TN: 222,925
    TP: 808
    FN: 304
    FP: 23,243

Verifying target days in aggregated data...
  20220724: 562 blocks  |  fires=3  TP=2  FP=45  FN=1  TN=514
  20210719: 562 blocks  |  fires=49  TP=45  FP=135  FN=4  TN=378

  Block dot size (s): 137.2

Generating two-panel aggregated confusion map...
  Saved: figures/fig_confusion_map_agg5x5_two_days.png
  Saved: figures/fig_confusion_map_agg5x5_two_days.pdf

SUMMARY — AGGREGATED TWO-DAY CONFUSION MAP
  Model     : model_ignition_agg5x5_smotebaseline.pkl
  Threshold : 0.7398
  Block size: 5